# 🔬 RAG Pipeline Experiments
### Hybrid Retrieval · CrossEncoder Reranking · Groq LLM
A full end-to-end RAG pipeline over AI/ML research papers — from PDF loading to answer generation.

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

c:\Users\Vishal\Desktop\Rag_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## ⚙️ Setup — Imports & Environment
Load all required libraries and set the Groq API key from the `.env` file.

In [ ]:
from dotenv import load_dotenv
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")

## 📄 1. Load Research Papers
Use `PyMuPDFLoader` to load all PDF research papers from the local directory. Each page becomes a separate `Document` object with metadata (title, page number, source).

In [ ]:

from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path

docs = []
pdf_dir = Path("data/selected_papers")
for pdf_file in pdf_dir.glob("*.pdf"):
    try:
        loader = PyMuPDFLoader(str(pdf_file))
        docs.extend(loader.load())
    except Exception as e:
        print(f"Failed to load {pdf_file.name}: {e}")

print(f"✓ Loaded {len(docs)} pages from {len(list(pdf_dir.glob('*.pdf')))} PDFs")


✓ Loaded 4573 pages from 238 PDFs


## 📚 2. Add AI Concept Definitions
Load curated AI/LLM term definitions from a JSON file and convert them into `Document` objects. These are merged with the paper pages to enrich the knowledge base with foundational concepts.

In [ ]:

import json
from langchain_core.documents import Document

# Load AI/LLM definitions and merge into docs
definitions_path = r"data/ai_definitions.json"

with open(definitions_path, "r", encoding="utf-8") as f:
    definitions = json.load(f)

# Convert each definition into a LangChain Document
definition_docs = []
for entry in definitions:
    text = f"{entry['term']}: {entry['definition']}"
    doc = Document(
        page_content=text,
        metadata={
            "source": "ai_definitions.json",
            "title": "AI & LLM Definitions",
            "term": entry["term"],
            "page": 0
        }
    )
    definition_docs.append(doc)

# Merge into docs list
docs.extend(definition_docs)

print(f"✓ Added {len(definition_docs)} definitions")
print(f"✓ Total docs (pages + definitions): {len(docs)}")


✓ Added 101 definitions
✓ Total docs (pages + definitions): 4674


## ✂️ 3. Document Chunking + BM25 Index
Split all documents into overlapping chunks using `RecursiveCharacterTextSplitter` (chunk=600, overlap=120). Simultaneously build a **BM25 sparse index** over the chunks for keyword-based retrieval. Chunks are also saved to `chunks.json`.

In [6]:

import json
import re

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=120,
    separators=["\n\n", "\n", ".", " ", ""]
)   
chunks = text_splitter.split_documents(docs)

from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\w+", text.lower())

bm25_corpus = [tokenize(chunk.page_content) for chunk in chunks]

# Build BM25 index
bm25 = BM25Okapi(bm25_corpus)

print("BM25 index built successfully")

# Save chunks to JSON file
chunks_data = []
for chunk in chunks:
    chunks_data.append({
        'content': chunk.page_content,
        'metadata': chunk.metadata
    })

with open('chunks.json', 'w', encoding='utf-8') as f:
    json.dump(chunks_data, f, indent=2, ensure_ascii=False)

print(f"✓ Saved {len(chunks)} chunks to chunks.json")


BM25 index built successfully
✓ Saved 34759 chunks to chunks.json
✓ Saved 34759 chunks to chunks.json


## 🧲 4. Embeddings + FAISS Vector Index
Load `BAAI/bge-large-en-v1.5` on GPU to generate dense embeddings. Build a **FAISS vector index** from all chunks and save it locally. On subsequent runs, load the saved index directly to skip re-embedding.

In [9]:

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import torch

# Check if GPU is available
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    device = "cuda"
else:
    print("No GPU found, using CPU")
    device = "cpu"

# Clear any leftover GPU memory before loading model
if device == "cuda":
    torch.cuda.empty_cache()

# Create embedding model - reduced batch size for 6GB GPU with large corpus
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32,  # Reduced from 128 to avoid CUBLAS OOM on 6GB GPU
    }
)

print(f"✓ Embedding model loaded on {device} with batch_size=32")


CUDA available: True
GPU device: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU memory: 6.0 GB


C:\Users\Vishal\AppData\Local\Temp\ipykernel_27536\719242742.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4382.20it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4382.20it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  

✓ Embedding model loaded on cuda with batch_size=32


In [10]:
# Create FAISS vector store from documents
db = FAISS.from_documents(chunks, embedding_model)
print("Vector store has been created successfully!")

# Save the vector store locally
db.save_local("faiss_index")

print(f"FAISS index created and saved with {len(chunks)} documents")

Vector store has been created successfully!
FAISS index created and saved with 34759 documents
FAISS index created and saved with 34759 documents


In [11]:
# Load the saved FAISS index
db = FAISS.load_local("faiss_index", embedding_model, allow_dangerous_deserialization=True)

print(f"FAISS index loaded successfully!")

FAISS index loaded successfully!


## 🔍 5. Hybrid Retrieval (FAISS + BM25 + RRF)
Run both **FAISS vector search** (semantic) and **BM25** (keyword) in parallel, then merge results using **Reciprocal Rank Fusion (RRF)**. This combines the strengths of both approaches for more robust retrieval.

In [ ]:
def hybrid_retrieval(query, k_vector=80, k_bm25=80, rrf_k=30, top_n=40):
    """
    Hybrid retrieval using Vector + BM25 merged with Reciprocal Rank Fusion (RRF).
    rrf_k: RRF constant (typical values 60).
    """
    # --- Vector search (ranked list) ---
    vector_results = db.similarity_search(query, k=k_vector)

    # --- BM25 search (ranked list) ---
    tokenized_query = tokenize(query)
    bm25_scores = bm25.get_scores(tokenized_query)
    top_bm25_indices = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True
    )[:k_bm25]
    bm25_results = [chunks[i] for i in top_bm25_indices]

    # --- Helper to create stable key for a document ---
    def doc_key(doc):
        md = getattr(doc, "metadata", {}) or {}
        # Prefer id/source/title+page if available, else fallback to content hash
        key = md.get("id") or md.get("source") or md.get("title")
        if key:
            return f"{key}|{md.get('page','')}"
        return str(hash(doc.page_content))

    # --- Compute RRF scores ---
    scores = {}
    doc_map = {}

    # vector_results ranks (i starts at 0 => rank = i+1)
    for i, doc in enumerate(vector_results):
        k = doc_key(doc)
        doc_map[k] = doc  # keep representative doc
        scores[k] = scores.get(k, 0.0) + 1.0 / (rrf_k + (i + 1))

    # bm25_results ranks
    for i, doc in enumerate(bm25_results):
        k = doc_key(doc)
        # If doc not seen yet, keep representative
        if k not in doc_map:
            doc_map[k] = doc
        scores[k] = scores.get(k, 0.0) + 1.0 / (rrf_k + (i + 1))

    # --- Sort by fused score and return top_n documents ---
    ranked_keys = sorted(scores.keys(), key=lambda k: scores[k], reverse=True)
    ranked_docs = [doc_map[k] for k in ranked_keys][:top_n]

    return ranked_docs

In [31]:
query = "What training objective does BERT use?"
# query = "CrossCheck alerts operators before they trigger network outages."
results = hybrid_retrieval(query)

In [32]:
# Reranking
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-large",device=device)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5005.79it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 🎯 6. CrossEncoder Reranking
Load `BAAI/bge-reranker-large` CrossEncoder to rerank the top retrieved documents. Unlike bi-encoders, a CrossEncoder scores each (query, document) pair together — giving much more accurate relevance scores. Top-5 docs are selected for the LLM.

In [33]:

pairs = [[query, doc.page_content] for doc in results]

scores = reranker.predict(pairs)

reranked = sorted(
    zip(results, scores),
    key=lambda x: x[1],
    reverse=True
)
top_docs = [doc for doc, score in reranked[:5]]

## 🤖 7. LLM Answer Generation (Groq)
Build a LangChain LCEL chain with `llama-3.1-8b-instant` via the Groq API. The top-5 reranked docs are formatted as context, and the LLM generates a grounded answer with paper title and page citations.

In [16]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize the LLM
llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.1-8b-instant")

# Create the prompt
prompt = PromptTemplate.from_template("""
You are a research assistant.

Answer the question using ONLY the context below.
If the answer cannot be found in the context, say "Not found in provided papers."

Context:
{context}

Question:
{question}

Return:

Answer:
<explanation>

Sources:
- Paper: <paper title> | Page: <page number>
""")
# Create the chain using LCEL (LangChain Expression Language)
chain = prompt | llm | StrOutputParser()

In [34]:
query = 'What is the purpose of evaluation datasets in RAG systems?'
results = hybrid_retrieval(query)

pairs = [[query, doc.page_content] for doc in results]

scores = reranker.predict(pairs)

reranked = sorted(
    zip(results, scores),
    key=lambda x: x[1],
    reverse=True
)

top_docs = [doc for doc, score in reranked[:5]]
context = "\n\n".join([
    f"Paper: {doc.metadata.get('title')} | Page: {doc.metadata.get('page')}\n{doc.page_content}"
    for doc in top_docs
])

## 🧪 8. End-to-End Query Test
Run a full query through the entire pipeline: hybrid retrieval → CrossEncoder reranking → LLM answer generation. Inspect the final response with source citations.

In [35]:
response = chain.invoke({
    "context": context, 
    "question": query
})

In [36]:
print(response)

The purpose of evaluation datasets in RAG systems is to assess the models' strengths and weaknesses, as well as to provide a useful metric for model selection in practical RAG deployment and for building RAG-specialized models.

Source: 
- Paper: LIT-RAGBench: Benchmarking Generator Capabilities of Large Language Models in Retrieval-Augmented Generation | Page: 1
